In [ ]:
import numpy as np
import os
import sys
import glob
import shutil
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.wcs import WCS
from astropy.coordinates import SkyCoord
import astropy.units as u
from matplotlib.patches import Circle
from astropy.table import Table
import time
import xarray as xr

from astropy.nddata import Cutout2D
from matplotlib.patches import Circle
from Functions import *
#sys.path.append('/d/ret1/Taylor/jupyter_notebooks/Research/slug2')  # Path to slug2 directory
sys.path.append('/project/galaxies/tjuchau/software/Slug/slug2')
import re
#slug_path = "/d/ret1/Taylor/jupyter_notebooks/Research/slug2/bin/slug"
slug_path = "/project/galaxies/tjuchau/software/Slug/slug2/src/slug"
import slugpy
#wd = '/d/ret1/Taylor/jupyter_notebooks/Research'
wd = '/project/galaxies/tjuchau'
slug_output_dir = '/project/galaxies/tjuchau/data_files/misc_data/slug_outputs'
os.chdir('/project/galaxies/tjuchau/projects/EW_vs_Age/modeling_files')


In [ ]:
def write_slug_input(model_name, mass,dt = 1e6, N=1000, tracks = "mist_2016_vvcrit_00", spec_synth = 'kurucz'):
    '''Replication of Alex's model'''
    input_file = f"{model_name}.slugin"
    
    with open(input_file, 'w') as f:
        f.write(f'model_name {model_name}\n')
        f.write(f'out_dir {slug_output_dir}\n') #TJ this is where the output files will be written to
        f.write(f'verbosity 2\n') #TJ level of printed outputs while running (0=only warnings/errors) (1=some outputs) (2=lots of outputs)
        ##################################################################
        # Parameters controlling simulation execution and physical model #
        ##################################################################
        f.write(f'sim_type cluster\n') #TJ must be either galaxy or cluster (defaults to galaxy)
        f.write(f'n_trials {N}\n') #TJ total number of model clusters to run
        #f.write(f'checkpoint_interval = 100\n') #TJ create checkpoint after this many trials (default to no checkpointing)
        f.write(f'time_step {dt}\n') #TJ simulation runs for 1million years before computing new values
        #f.write(f'start_time 1.0e6\n') #TJ default start time is the same as timestep
        f.write(f'end_time 1.0e7\n') #TJ how long does each simulation run for in years
        #f.write(f'sfr 0.001\n') #TJ star formation rate, ignored for sim types = cluster
        #f.write(f'sfh sfh.txt\n') #TJ star formation history, ignored for sim types = cluster
        f.write(f'cluster_mass {mass}\n') #TJ cluster mass in solar masses, ignored for sim type = galaxy
        #f.write(f'redshift 0\n') #TJ defaults to 0
        ##################################################################
        # Parameters controlling simulation outputs #
        ##################################################################
        f.write(f'out_cluster 1\n') #TJ output cluster properties? default = 1
        f.write(f'out_cluster_phot 1\n') #TJ output cluster photometry? (must specify filters also)
        f.write(f'out_cluster_spec 1\n') #TJ output cluster spectroscopy? *adds significant computation time*
        f.write(f'out_cluster_yield 1\n') #TJ output cluster nucleosynthesis yields?
        #f.write(f'out_integrated 1\n') #TJ output integrated properties of galaxy? ignored for sim types = cluster
        #f.write(f'out_integrated_phot 1\n') #TJ output integrated photometry of galaxy? ignored for sim types = cluster
        #f.write(f'out_integrated_spec 1\n') #TJ output integrated spectroscopy of galaxy? ignored for sim types = cluster
        #f.write(f'out_integrated_yield 1\n') #TJ output integrated chemical yields of galaxy? ignored for sim types = cluster
        f.write(f'output_mode ascii\n') #TJ can be either binary, ascii, or fits
        #####################################################################
        # Parameters controlling the physical models used for stars         #
        #####################################################################
        #f.write(f'imf lib/imf/chabrier.imf\n') #TJ what imf to use? defaults to chabrier 2001
        #f.write(f'cmf lib/cmf/slug_default.cmf\n') #TJ cluster mass function for galaxies, ignored for sim types = cluster
        f.write(f'clf lib/clf/nodisrupt.clf\n') #TJ cluster lifetime Default: lib/clf/slug_default.clf (dN/dt ~ t^-1.9)
        f.write(f'tracks {tracks}\n') #TJ choose the stellar track. Defaults to geneva_2013_vvcrit_00
        f.write(f'atmospheres lib/atmospheres\n') #TJ directory of the stellar atmospheres
        f.write(f'specsyn_mode {spec_synth}\n') #TJ Spectral synthesis mode, describing which models to use for stellar atmospheres allowed values below
        # -- planck (treat stars as blackbodies)
        # -- kurucz (use Kurucz atmospheres, as compiled by Lejeune+ 1997)
        # -- kurucz+hillier (use Hillier models for WR stars, kurucz for all others)
        # -- kurucz+pauldrach (use Pauldrach models for OB stars, kurucz for others)
        # -- sb99 (emulate starburst99 -- Pauldrach for OB stars, Hillier for WR stars, kurucz for others) This is the default value
        f.write(f'clust_frac 1.0\n') #TJ fraction of stars born in clusters (always 1.0 for sim types = cluster)
        f.write(f'min_stoch_mass 0.08\n') #TJ minimum stochastically sampled mass. Everything below is considered to be continuously sampled
        #f.write(f'metallicity       1.0\n') #TJ metalicity function. If tracks is specified, this should be omitted
        #####################################################################
        # Parameters controlling extinction                                 #
        #####################################################################
        f.write(f'A_V lib/avdist/slug_default.av\n') #TJ set extinction function
        f.write(f'extinction_curve lib/extinct/MW_EXT_SLUG.dat\n') #TJ shape of extinction curve
        f.write(f'nebular_extinction_factor lib/avdist/neb_factor_default.av\n') #TJ use a different extinction law for nebulae
        #####################################################################
        # Parameters controlling nebular emission                           #
        #####################################################################
        f.write(f'compute_nebular 1\n') #TJ compute nebular emission specifically?
        #f.write(f'atomic_data lib/atomic\n') #TJ atomic information, defaults to lib/atomic
        #f.write(f'nebular_no_metals 0\n') #TJ 1 would be to turn off nebular metal emission (includes He), 0 means leave metals on
        #f.write(f'nebular_den 1.0e2\n') #TJ hydrogen density (default is 100)
        #f.write(f'nebular_temp -1.0\n') #TJ nebular temperature, default is -1, if negative, temp will be calculated from cloudy
        f.write(f'nebular_logU -2.5\n') #TJ logU representing ionization parameter
        f.write(f'nebular_phi 0.73\n') #TJ fraction of ionizing photons that are absorbed by Hydrogen atoms
        #############################################
        # Parameters describing photometric filters #
        #############################################
        f.write(f'phot_bands JWST_F150W, JWST_F187N, JWST_F300M, QH0\n') #TJ list of filters for photometric results
        #f.write(f'filters lib/filters\n') #TJ directory for filter information to be read from (defaults to lib/filters)
        #f.write(f'phot_mode Lnu\n') #TJ what units should the photometry results print in? (defaults to Lnu)
        ############################################
        # Parameters controlling yield calculation #
        ############################################
        f.write(f'yield_dir lib/yields\n') #TJ directory for yield files
        
        # are available:
        # 
        
        # 
        f.write(f'yield_mode sukhbold16+karakas16+doherty14\n') #TJ Model to use for yield calculation. Currently the following models accepted:
        # -- sukhbold16 = Solar metallicity type II SN yields from Sukhbold et al. (2016, ApJ, 821, 38); no other yields
        # -- # karakas16+doherty14 = metallicity-dependent AGB star yields from Karakas & Lugaro (2016, ApJ, 825, 26), and super- 
        #                                                                                 AGB star yields from Doherty+ (2014, MNRAS, 437, 195)
        # -- sukhbold16+karakas16+doherty14 = sukhbold16 used for SNII, karakas16+doherty14 for AGB
        f.write(f'\n')
        wd = os.getcwd().split('medbow')[-1]

    return wd+f'/{input_file}'

def read_slug_spec(spec_file):
    with open(spec_file, 'r') as f:
        lines = f.readlines()
    
    col_names = lines[0].strip().split()
    n_cols = len(col_names)
    
    data_entries = []
    for line in lines[2:]:
        if line.startswith('---------'):
            continue
        
        entries = line.strip().split()
        if len(entries) < n_cols:
            entries += ['nan'] * (n_cols - len(entries))
        
        data_entries.append(entries)
    
    return np.array(data_entries, dtype=float)

def read_all_files(model_name, slug_output_dir = '/project/galaxies/tjuchau/data_files/misc_data/slug_outputs'):
    '''
    Reads all SLUG output .txt files for a given model_name into a dictionary of numpy arrays.
    Keys are column names, values are data arrays.
    '''
    output = {}
    files = glob.glob(f'{slug_output_dir}/{model_name}_cluster*.txt')
    
    for file in files:
        #print(f'reading {file}')
        if file[-5]=='d':
            #print(f'skipping {file}')
            continue
        elif file[-9:] == '_spec.txt':
            continue
        with open(file, 'r') as f:
            lines = f.readlines()
        
        # Get column names
        col_names = lines[0].strip().split()
        n_cols_names = len(col_names)
        
        # Parse data lines
        data_entries = []
        
        for line in lines[2:]:
            stripped = line.strip()
            if line.startswith('---------'):
                continue
            entries = stripped.split()
            
            # Pad missing entries with 'nan'
            if len(entries) < n_cols_names:
                entries += ['nan'] * (n_cols_names - len(entries))
            data_entries.append(entries)
        
        # Skip empty files gracefully
        if not data_entries:
            continue
        
        # Convert to numpy array of strings first
        data_array = np.array(data_entries, dtype='U20')
        
        # Convert each column individually to float if possible, else keep as string
        for i, name in enumerate(col_names):
            col = data_array[:, i]
            try:
                col_converted = col.astype(float)
            except ValueError:
                col_converted = col  # keep as string if conversion fails
            
            if name in output:
                #print(f"Warning: Duplicate column name '{name}' found. Overwriting previous value.")
                print('', end = '\r')
            output[name] = col_converted
            
    return output

def compute_paalpha_ew(wavelength, flux,
                      line_center=18756.0,
                      line_window=200,     # +/- Angstrom around line
                      cont_window=500):    # continuum window
    
    # --- Define regions ---
    line_mask = (wavelength > line_center - line_window) & \
                (wavelength < line_center + line_window)
    
    cont_mask = (
        ((wavelength > line_center - cont_window) & 
         (wavelength < line_center - line_window)) |
        ((wavelength > line_center + line_window) & 
         (wavelength < line_center + cont_window))
    )
    
    if np.sum(cont_mask) < 5:
        return np.nan
    
    # --- Continuum estimate ---
    cont_level = np.median(flux[cont_mask])
    
    if cont_level <= 0:
        return np.nan
    
    # --- Sort line region ---
    wl_line = wavelength[line_mask]
    fl_line = flux[line_mask]
    
    sort_idx = np.argsort(wl_line)
    wl_line = wl_line[sort_idx]
    fl_line = fl_line[sort_idx]
    
    # --- Compute EW ---
    integrand = (fl_line - cont_level) / cont_level
    ew = np.trapezoid(integrand, wl_line)
    
    return ew

def build_dataset_from_slug(spec_file, phot_data_dict):
    
    data = read_slug_spec(spec_file)
    
    trial = data[:,0].astype(int)
    time = data[:,1]
    wavelength = data[:,2]
    
    flux_total = data[:,3] + data[:,4]
    
    unique_trials = np.unique(trial)
    unique_times = np.unique(time)
    unique_wl = np.unique(wavelength)
    
    Nt = len(unique_trials)
    Tt = len(unique_times)
    Nw = len(unique_wl)
    
    spectrum = np.full((Nt, Tt, Nw), np.nan)
    
    # --- Fill cube ---
    for i, tr in enumerate(unique_trials):
        for j, t in enumerate(unique_times):
            mask = (trial == tr) & (time == t)
            
            wl = wavelength[mask]
            fl = flux_total[mask]
            
            if len(wl) == 0:
                continue
            
            sort_idx = np.argsort(wl)
            wl = wl[sort_idx]
            fl = fl[sort_idx]
            
            # map onto global wavelength grid
            idx = np.searchsorted(unique_wl, wl)
            spectrum[i, j, idx] = fl
    
    # --- Compute EW ---
    ew = np.zeros((Nt, Tt))
    
    for i in range(Nt):
        for j in range(Tt):
            wl = unique_wl
            fl = spectrum[i, j, :]
            
            if np.all(np.isnan(fl)):
                ew[i,j] = np.nan
            else:
                ew[i,j] = compute_paalpha_ew(wl, fl)
    
    ds = xr.Dataset(
        {
            "spectrum": (["trial","time","wavelength"], spectrum),
            "ew_spec": (["trial","time"], ew),
        },
        coords={
            "trial": unique_trials,
            "time": unique_times,
            "wavelength": unique_wl
        }
    )
    
    return ds

def find_model_names(base_string, slug_output_dir):
    files = glob.glob(f"{slug_output_dir}/{base_string}*_summary.txt")
    
    model_names = []
    masses = []
    
    for f in files:
        name = os.path.basename(f)
        
        # extract mass from string
        match = re.search(r'_m_(\d+)_summary', name)
        if match:
            m = int(match.group(1))
            model_name = name.replace('_summary.txt', '')
            
            model_names.append(model_name)
            masses.append(m)
    
    return model_names, masses

def build_single_model_dataset(model_name, slug_output_dir):
    
    # -------- Read summary (metadata) --------
    summary_file = f"{slug_output_dir}/{model_name}_summary.txt"
    
    with open(summary_file, 'r') as f:
        lines = f.readlines()
    
    # crude but reliable parsing
    summary_dict = {}
    summary_targets = ['time_step', 'n_trials', 'cluster_mass']
    for line in lines[1:]:
        if np.any([x in line for x in summary_targets]):
            key, val = line.split()
            summary_dict[key.strip()] = val.strip()
    
    dt = float(summary_dict.get('time_step', np.nan))
    n_trials = int(summary_dict.get('n_trials', 1))
    mass = float(summary_dict.get('cluster_mass', np.nan))
    
    # -------- Read all non-spec files --------
    data_dict = read_all_files(model_name, slug_output_dir)
    
    # REQUIRED columns
    trial = data_dict['UniqueID'].astype(int)
    time = data_dict['Time']
    
    unique_trials = np.unique(trial)
    unique_times = np.unique(time)
    
    Nt = len(unique_trials)
    Tt = len(unique_times)
    
    # -------- Build structured arrays --------
    variables = {}
    
    for key, arr in data_dict.items():
        
        if len(arr) != len(trial):
            continue
        
        grid = np.full((Nt, Tt), np.nan)
        
        for i, tr in enumerate(unique_trials):
            for j, t in enumerate(unique_times):
                mask = (trial == tr) & (time == t)
                
                if np.any(mask):
                    grid[i, j] = arr[mask][0]
        
        variables[key] = (["trial", "time"], grid)
    
    # -------- Build dataset --------
    ds = xr.Dataset(
        variables,
        coords={
            "trial": unique_trials,
            "time": unique_times
        }
    )
    
    # -------- Attach metadata --------
    ds = ds.assign_coords(mass=mass)
    ds.attrs['dt'] = dt
    ds.attrs['model_name'] = model_name.split('_m_')[0]
    
    return ds

def add_spectra_to_dataset(ds, model_name, slug_output_dir):
    
    spec_file = f"{slug_output_dir}/{model_name}_cluster_spec.txt"
    
    data = read_slug_spec(spec_file)
    
    trial = data[:,0].astype(int)
    time = data[:,1]
    wavelength = data[:,2]
    flux = data[:,3] + data[:,4]
    
    unique_wl = np.unique(wavelength)
    
    Nt = len(ds.trial)
    Tt = len(ds.time)
    Nw = len(unique_wl)
    
    spectrum = np.full((Nt, Tt, Nw), np.nan)
    
    for i, tr in enumerate(ds.trial.values):
        for j, t in enumerate(ds.time.values):
            
            mask = (trial == tr) & (time == t)
            
            wl = wavelength[mask]
            fl = flux[mask]
            
            if len(wl) == 0:
                continue
            
            idx = np.searchsorted(unique_wl, wl)
            spectrum[i, j, idx] = fl
    
    ds["spectrum"] = (["trial","time","wavelength"], spectrum)
    ds = ds.assign_coords(wavelength=unique_wl)
    
    return ds

def check_if_job_finished(job_directory, job_title):
    """
    Check if a SLURM job has finished by inspecting .out and .err files.
    
    Prints status inline and returns True when job is done.
    """
    
    print(f"checking if job {job_title} is done               ", end='\r')
    
    out_file = os.path.join(job_directory, f"{job_title}.out")
    err_file = os.path.join(job_directory, f"{job_title}.err")
    
    # --- Check existence first ---
    if not (os.path.exists(out_file) and os.path.exists(err_file)):
        return False
    
    # --- Check non-zero size ---
    size_out_1 = os.path.getsize(out_file)
    size_err_1 = os.path.getsize(err_file)
    
    if size_out_1 == 0 or size_err_1 == 0:
        return False
    
    # --- Wait to ensure writing is finished ---
    time.sleep(5)
    
    size_out_2 = os.path.getsize(out_file)
    size_err_2 = os.path.getsize(err_file)
    
    # --- Ensure files are no longer growing ---
    if size_out_1 == size_out_2 and size_err_1 == size_err_2:
        print(f"job {job_title} finished                             ")
        return True
    
    return False

def add_ew_from_filters(ds):
    """
    Compute Paα continuum, line flux, and EW from JWST filters
    and convert continuum to L_lambda (erg/s/Å) for consistency with spectra.
    """
    
    required = {"JWST_F150W", "JWST_F300M", "JWST_F187N"}
    if not required.issubset(ds.data_vars):
        missing = required - set(ds.data_vars)
        raise ValueError(f"Missing required variables: {missing}")
    
    # --- Extract filters (L_nu) ---
    f150 = ds["JWST_F150W"]
    f300 = ds["JWST_F300M"]
    f187n = ds["JWST_F187N"]
    
    # --- Wavelengths (Å) ---
    lambda150 = 15000.0
    lambda187 = 18750.0
    lambda300 = 30000.0
    
    # --- Speed of light (Å/s) ---
    c = 2.99792458e18
    
    # --- Convert to L_lambda ---
    f150_lam = f150 * c / (lambda150**2)
    f300_lam = f300 * c / (lambda300**2)
    f187n_lam = f187n * c / (lambda187**2)
    
    # --- Continuum interpolation in L_lambda ---
    slope = (f300_lam - f150_lam) / (lambda300 - lambda150)
    continuum = f150_lam + slope * (lambda187 - lambda150)
    
    # --- Line flux (still in L_lambda units) ---
    paalpha_flux = f187n_lam - continuum
    
    # --- EW (unchanged conceptually) ---
    f187n_width = 200.0  # Å
    ew = f187n_width * (paalpha_flux / continuum)
    
    ds = ds.assign(
        ew_phot=ew,
        phot_cont=continuum   # now in L_lambda
    )
    
    return ds

def add_ew_from_spec(ds,
                     line_center=18756.0,
                     line_window=200,
                     cont_window=500):
    """
    Compute Paα EW from spectra and add it to the dataset.
    
    Requires:
        ds["spectrum"] with dims (..., wavelength)
        ds["wavelength"] coordinate
    """

    if "spectrum" not in ds:
        raise ValueError("Dataset must contain 'spectrum'")
    
    if "wavelength" not in ds.coords:
        raise ValueError("Dataset must contain 'wavelength' coordinate")

    wavelength = ds["wavelength"]

    # --- Define masks (1D, along wavelength) ---
    line_mask = (
        (wavelength > line_center - line_window) &
        (wavelength < line_center + line_window)
    )
    
    cont_mask = (
        ((wavelength > line_center - cont_window) &
         (wavelength < line_center - line_window)) |
        ((wavelength > line_center + line_window) &
         (wavelength < line_center + cont_window))
    )

    # --- Core EW function (operates on 1D arrays) ---
    def compute_ew_1d(flux_1d, wl_1d):
        
        fl_line = flux_1d[line_mask]
        wl_line = wl_1d[line_mask]
        fl_cont = flux_1d[cont_mask]
        
        
        cont_level = np.nanmedian(fl_cont)
        
        if not np.isfinite(cont_level) or cont_level <= 0:
            print('NOPE!')
            return np.nan
        
        # sort line region
        sort_idx = np.argsort(wl_line)
        wl_line = wl_line[sort_idx]
        fl_line = fl_line[sort_idx]
        
        integrand = (fl_line - cont_level) / cont_level
        
        return np.trapezoid(integrand, wl_line), cont_level

    # --- Apply across all non-wavelength dims ---
    ew, cont = xr.apply_ufunc(
        compute_ew_1d,
        ds["spectrum"],
        wavelength,
        input_core_dims=[["wavelength"], ["wavelength"]],
        output_core_dims=[[], []],  # scalar output
        vectorize=True,
        dask="parallelized",
        output_dtypes=[float, float],
    )

    # --- Add to dataset ---
    ds = ds.assign(ew_spec=ew)
    ds = ds.assign(spec_cont = cont)
    return ds

def build_dataset_from_base(base_string, slug_output_dir, overwrite=False):
    
    model_names, masses = find_model_names(base_string, slug_output_dir)

    base_path = os.path.join(slug_output_dir, f"{base_string}.nc")
    
    if os.path.exists(base_path) and not overwrite:
        print(f"WARNING: {base_path} already exists. Generating new filename...")
        
        i = 1
        while True:
            new_path = os.path.join(slug_output_dir, f"{base_string}_{i}.nc")
            if not os.path.exists(new_path):
                print(f"Saving to: {new_path}")
                output_path = new_path
                break
            i += 1
    else:
        output_path = base_path

    datasets = []
    
    for model_name, m in zip(model_names, masses):
        print(f"Processing {model_name}                       ")
        
        ds = build_single_model_dataset(model_name, slug_output_dir)
        
        # optional: add spectra
        ds = add_spectra_to_dataset(ds, model_name, slug_output_dir)
        
        ds = ds.expand_dims({"mass": [m]})
        
        datasets.append(ds)
    
    # -------- Combine all masses --------
    ds_all = xr.concat(datasets, dim="mass")
    ds_all = add_ew_from_filters(ds_all)
    ds_all = add_ew_from_spec(ds_all)
    ds_all.to_netcdf(output_path)
    
    return ds_all

def run_slug_and_compile(base_string, mass_array = [200, 2000,5000,10000,20000,50000],
dt = 1e6, N=1000, tracks = "mist_2016_vvcrit_00", spec_synth = 'kurucz'):
    jobs = []
    job_directory = '/cluster/medbow/project/galaxies/tjuchau/projects/EW_vs_Age/modeling_files'
    for m in np.sort(mass_array)[::-1]:
        model_name = f'{base_string}_m_{m}'
        slug_input_file = write_slug_input(model_name, m, dt = dt, N=N, tracks = tracks, spec_synth = spec_synth)
        
        run_command_on_ARCC(job_directory,
        model_name, f"conda run -n Modeling {slug_path} {slug_input_file}", 
        cpus = 1 if m < 10001 else 5, memory=8, fail_notification=True, finish_notification=False, time=8, conda = "Modeling")
        jobs.append(model_name)
    finished = {job: False for job in jobs}

    while not all(finished.values()):
        for job in jobs:
            if not finished[job]:
                finished[job] = check_if_job_finished(job_directory, job)
                print(f'checking if {job} is done                       ', end = '\r')
        time.sleep(5) 
    
    data = build_dataset_from_base(base_string, slug_output_dir)
    
    return data

def plot_all_trials(ds, y_var="ew_spec",
                    max_cols=3,
                    show_median=True,
                    alpha=None):
    """
    Plot all trials vs time for each mass bin from an xarray Dataset.

    Parameters
    ----------
    ds : xarray.Dataset
        Must contain dims: ('mass', 'trial', 'time')
    
    y_var : str
        Variable to plot on y-axis (default: 'paalpha_ew_spec')
    
    max_cols : int
        Number of subplot columns
    
    show_median : bool
        Whether to overplot median across trials
    
    alpha : float or None
        Transparency for individual trials (auto if None)
    """

    if y_var not in ds:
        raise ValueError(f"{y_var} not found in dataset")

    masses = np.sort(ds.mass.values)
    n_masses = len(masses)

    # --- layout ---
    n_cols = max_cols
    n_rows = int(np.ceil(n_masses / n_cols))

    fig, axes = plt.subplots(n_rows, n_cols,
                             figsize=(5*n_cols, 3*n_rows),
                             sharex=True, sharey=True)

    axes = np.array(axes).flatten()

    # --- auto alpha scaling ---
    n_trials = ds.sizes.get("trial", 1)
    if alpha is None:
        alpha = 0.7 if n_trials < 20 else 0.1

    # --- loop over mass bins ---
    for i, m in enumerate(masses):
        ax = axes[i]

        subset = ds.sel(mass=m)

        # --- plot all trials at once (xarray magic) ---
        subset[y_var].plot.line(
            x="time",
            hue="trial",
            ax=ax,
            alpha=alpha,
            add_legend=False
        )

        # --- median ---
        if show_median:
            median = subset[y_var].median(dim="trial", skipna=True)
            median.plot(ax=ax, color="black", linewidth=2, label="Median")

        # --- formatting ---
        ax.set_title(f"Mass = {m}")
        #ax.set_xscale("log")
        ax.set_xlabel("Age (yr)")
        ax.set_ylabel(y_var)

    # --- hide unused panels ---
    for j in range(i+1, len(axes)):
        axes[j].set_visible(False)

    plt.tight_layout()
    plt.show()


m=20000
dt = 1e6
N=10
job_directory = '/cluster/medbow/project/galaxies/tjuchau/projects/EW_vs_Age/modeling_files'
base_string = '00Test01'
model_name = f'{base_string}_m_{m}'
slug_input_file = write_slug_input(model_name, m, dt = dt, N=N)

run_command_on_ARCC(job_directory,
model_name, f"conda run -n Modeling {slug_path} {slug_input_file}", 
cpus = 1 if m < 10001 else 5, memory=8, fail_notification=True, finish_notification=False, time=8, conda = "Modeling")

In [ ]:
#TJ make new dataset and compile. Takes several minutes

base_string = 'Full_functionality_test_dt6_testing'
ds = run_slug_and_compile(base_string, N=10, dt=1e6)
'''ds = build_dataset_from_base(base_string, slug_output_dir, overwrite=True)

ds.to_netcdf(f"{slug_output_dir}/{base_string}.nc")

base_string = 'Full_functionality_test_dt5'

ds = build_dataset_from_base(base_string, slug_output_dir, overwrite=True)

ds.to_netcdf(f"{slug_output_dir}/{base_string}.nc")
'''

In [ ]:

def plot_trial_diagnostics(ds, variables, custom=0, x_var=None, mass=2000, trials=None, n_trials=10):
    """
    Plot stacked diagnostics for a given mass:
    
    Top: ew_phot
    Middle: MaxStarMass
    Bottom: phot_cont + spec_cont
    
    Parameters
    ----------
    ds : xarray.Dataset
    mass : int/float
    trials : list or None
        Specific trial indices to plot
    n_trials : int
        Number of trials to randomly select if trials=None
    """
    colors = [
        "#1f77b4",  # Blue
        "#ff7f0e",  # Orange
        "#2ca02c",  # Green
        "#d62728",  # Red
        "#9467bd",  # Purple
        "#8c564b",  # Brown
        "#e377c2",  # Pink
        "#7f7f7f",  # Gray
        "#bcbd22",  # Olive
        "#17becf"   # Cyan
    ]
    subset = ds.sel(mass=mass)

    # --- choose trials ---
    all_trials = subset.trial.values

    if trials is None:
        if len(all_trials) <= n_trials:
            trials = all_trials
        else:
            trials = np.random.choice(all_trials, n_trials, replace=False)

    # --- figure setup ---
    fig, axes = plt.subplots(
        len(variables)+custom, 1, figsize=(8, int(custom+4*len(variables))),
        sharex=True
    )

    # --- color map for consistency across panels ---
    cmap = plt.cm.plasma
    norm = plt.Normalize(vmin=min(trials), vmax=max(trials))

    # --- loop through trials ---
    for i, tr in enumerate(trials):
        color = colors[i]

        trial_data = subset.sel(trial=tr)


        for a, ax in enumerate(variables):
            trial_data[variables[a]].plot.line(
                x="time" if x_var==None else x_var,
                ax=axes[a],
                color=color,
                alpha=0.8,
                add_legend=False
            )
            axes[a].set_ylabel(variables[a])
        if custom > 0:

            
            # Bottom: continua
            (trial_data["phot_cont"]/trial_data["spec_cont"]).plot.line(
                x="time",
                ax=axes[-1],
                color=color,
                linestyle='-',
                alpha=0.8,
                add_legend=False
                )



    axes[0].set_title(f"Mass = {mass}")

    # log time is usually better
    #ax3.set_xscale("log")

    # optional: log y where appropriate
    #ax2.set_yscale("log")

    # --- legend for continuum styles ---
    from matplotlib.lines import Line2D
    legend_elements = [
        Line2D([0], [0], color='black', linestyle='-', label='phot_cont'),
        Line2D([0], [0], color='black', linestyle='--', label='spec_cont')
    ]

    plt.tight_layout()
    plt.show()
base_string = 'Full_functionality_test_dt6'
ds = xr.load_dataset(f'{slug_output_dir}/{base_string}.nc')
plot_trial_diagnostics(ds,['ew_phot', 'MaxStarMass', "NumStar", 'JWST_F187N', "QH0"], custom = 1, mass=2000, trials=[23])

In [ ]:
base_string = 'Full_functionality_test_dt5'
ds = xr.load_dataset(f'{slug_output_dir}/{base_string}.nc')
plot_trial_diagnostics(ds,['ew_phot', 'MaxStarMass', "NumStar", 'JWST_F187N', "QH0"], custom = 1, mass=2000, trials=[1])

In [ ]:
base_string = 'Full_functionality_test_dt5'
ds = xr.load_dataset(f'/project/galaxies/tjuchau/data_files/misc_data/slug_outputs/{base_string}.nc')
#plot_all_trials(ds, y_var = 'ew_phot')
for i in range(30,40):

    (ds.sel(mass=2000, trial=i)['ew_spec']).plot.line(x='time', alpha=1)
 

In [ ]:
for i in range(1,11):
    test_dataset.sel(mass=10000, trial=i)['JWST_F187N'].plot(label=f'{i}')
plt.legend()
plt.show()
test_dataset.sel(mass=10000)['JWST_F187N'].plot.line(x='time', hue='trial', alpha=1)
plt.show()

In [ ]:
subset = ds.sel(
    mass=2000,
    trial=1,
    wavelength=slice(14500, 34000)
)
times = subset.time.values
cmap = plt.cm.rainbow
norm = plt.Normalize(vmin=times.min(), vmax=times.max())

plt.figure(figsize=(8,6))

for t in times:
    spec = subset.sel(time=t)['spectrum']
    
    plt.plot(
        subset.wavelength,
        spec,
        color=cmap(norm(t)),
        alpha=0.7,
    )

plt.xlabel("Wavelength (Å)")
plt.ylabel("Flux")
plt.title("Spectral evolution (Trial 1)")

plt.xscale('log')
plt.yscale('log')
plt.show()

In [ ]:
table_file = '/project/galaxies/tjuchau/data_files/Kiana_Cluster_Files/complete_updated_table.csv'
table = Table.read(table_file)
plt.scatter(table['best.stellar.age_m_star'][table['is_bad']==0], table['EW_187'][table['is_bad']==0], color = 'green', label = 'good')
plt.scatter(table['best.stellar.age_m_star'][table['is_bad']==1], table['EW_187'][table['is_bad']==1], color = 'red', label = 'maybe bad')
plt.scatter(table['best.stellar.age_m_star'][table['is_bad']==2], table['EW_187'][table['is_bad']==2], color = 'blue', label = 'bad')
#plt.yscale('log')
plt.xscale('log')
plt.title('not bad')
plt.legend()
plt.show()

table_file = '/project/galaxies/tjuchau/data_files/Kiana_Cluster_Files/complete_updated_table.csv'
table = Table.read(table_file)
plt.scatter(table['best.stellar.age_m_star'][table['is_bad']>0], table['EW_187'][table['is_bad']>0])
#plt.yscale('log')
plt.xscale('log')
plt.title('bads')
plt.show()

In [ ]:
fnu = (table[0]['F150W']*u.W/(u.Hz*u.m**2))
fnu*np.pi*(table[0]['best.universe.luminosity_distance']*u.m)**2

In [ ]:
table_file = '/project/galaxies/tjuchau/data_files/Kiana_Cluster_Files/complete_updated_table.csv'
table = Table.read(table_file)
table

In [ ]:
#ONCE I HAVE A WORKING LIBRARY
#################################################

import numpy as np
import numpy.ma as ma
from slugpy.cluster_slug import cluster_slug
import matplotlib.pyplot as plt

# ============================================================
# INPUT: your JWST photometry (replace with your catalog)
# ============================================================

# Example placeholders — replace with your data
# shape = (N_clusters,)
m150 = np.array(table[table['is_bad']==0]['F150W'])       # JWST F150W absolute mag
m187 = np.array(table[table['is_bad']==0]['F187N'])       # JWST F187N
m300 = np.array(table[table['is_bad']==0]['F300M'])       # JWST F300M

m150err = np.array(table[table['is_bad']==0]['F150W']*0.05)
m187err = np.array(table[table['is_bad']==0]['F187N']*0.05)
m300err = np.array(table[table['is_bad']==0]['F300M']*0.05)

# Mask invalid values if needed
valid = (~np.isnan(m150) &
         ~np.isnan(m187) &
         ~np.isnan(m300))

# Pack photometry
phot = np.vstack([m150[valid],
                  m187[valid],
                  m300[valid]]).T

photerr = np.vstack([m150err[valid],
                     m187err[valid],
                     m300err[valid]]).T

# ============================================================
# FILTER SET (JWST)
# ============================================================

filters = [
    'JWST_F150W',
    'JWST_F187N',
    'JWST_F300M'
]
f150_bw = (get_filter_data('F150W', aux_info=True)[2].to(u.AA)).value
f187_bw = (get_filter_data('F187N', aux_info=True)[2].to(u.AA)).value
f300_bw = (get_filter_data('F300M', aux_info=True)[2].to(u.AA)).value
bws = [f150_bw, f187_bw, f300_bw]
# ============================================================
# SAMPLE DENSITY (IMPORTANT!)
# This must match how you built your SLUG library
# ============================================================

def sample_density(physprop):
    logm = physprop[:,0]
    logt = physprop[:,1]

    sden = np.ones(len(logm))

    # example tapering (optional)
    sden[logm > 4] *= 10**(-(logm[logm > 4] - 4))
    sden[logt > 8] *= 10**(-(logt[logt > 8] - 8))

    return sden

# ============================================================
# LOAD YOUR CUSTOM LIBRARY
# ============================================================

cs = cluster_slug(
    photsystem='L_nu',   
    filters=filters,
    bw_phot='auto',
    sample_density=sample_density,
    libname='/project/galaxies/tjuchau/software/Slug/slug2/output/JWST_CLUSTER_LIB'
)

# ============================================================
# PRIORS (critical!)
# ============================================================

def priorfunc(physprop):
    logm = physprop[:,0]
    logt = physprop[:,1]
    av   = physprop[:,2]

    # Example:
    # flat in log age, flat in Av, p(M) ~ 1/M
    return 1.0 / np.exp(logm)

cs.priors = priorfunc

# ============================================================
# INFERENCE
# ============================================================

logm_grid, mpdf = cs.mpdf(0, phot, photerr, filters=filters)
logt_grid, tpdf = cs.mpdf(1, phot, photerr, filters=filters)

# Posterior means
m_mean = np.sum(logm_grid * mpdf * (logm_grid[1]-logm_grid[0]), axis=1)
t_mean = np.sum(logt_grid * tpdf * (logt_grid[1]-logt_grid[0]), axis=1)

# ============================================================
# OPTIONAL: best matches
# ============================================================

matches, dist = cs.bestmatch(phot, filters=filters, nmatch=5)

# ============================================================
# QUICK DIAGNOSTIC PLOT
# ============================================================

plt.scatter(t_mean, m_mean, s=5)
plt.xlabel("log Age [yr]")
plt.ylabel("log Mass [Msun]")
plt.title("Cluster Properties (bayesphot)")
plt.show()

In [ ]:

data = slugpy.read_cluster_phot(
    '/project/galaxies/tjuchau/software/Slug/slug2/output/JWST_CLUSTER_LIB'
)

slugpy.write_cluster(
    data,
    '/project/galaxies/tjuchau/software/Slug/slug2/output/JWST_CLUSTER_LIB',
    fmt='fits2'
)

In [ ]:
matches